## TrustyAI

<img src="https://trustyai.org/docs/main/_images/trustyai-logo-wide.svg" style="width:50%;">

---




In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Namespace
metadata:
  name: opendatahub
EOF

In [ ]:
%%bash
NAMESPACE=opendatahub
SERVICE=trustyai-service-operator-webhook-service

openssl req -x509 -newkey rsa:4096 \
  -keyout /tmp/webhook-server.key \
  -out /tmp/webhook-server.crt \
  -days 365 \
  -nodes \
  -subj "/CN=${SERVICE}.${NAMESPACE}.svc" \
  -addext "subjectAltName=DNS:${SERVICE},DNS:${SERVICE}.${NAMESPACE},DNS:${SERVICE}.${NAMESPACE}.svc,DNS:${SERVICE}.${NAMESPACE}.svc.cluster.local"

kubectl create secret tls webhook-server-cert \
  --cert=/tmp/webhook-server.crt \
  --key=/tmp/webhook-server.key \
  -n ${NAMESPACE}

Installation kustomize

In [ ]:
%%bash
curl -sSLo /tmp/kustomize.tar.gz https://github.com/kubernetes-sigs/kustomize/releases/download/kustomize%2Fv5.7.1/kustomize_v5.7.1_linux_amd64.tar.gz

tar -xzf /tmp/kustomize.tar.gz -C /tmp
sudo install -m 0755 /tmp/kustomize /usr/local/bin/kustomize

kustomize version

In [ ]:
%%bash
rm -rf trustyai-service-operator
git clone https://github.com/trustyai-explainability/trustyai-service-operator.git
cd trustyai-service-operator

OPERATOR_NAMESPACE=opendatahub
make manifest-gen NAMESPACE=$OPERATOR_NAMESPACE KUSTOMIZE=kustomize

In [ ]:
%%bash
cd trustyai-service-operator
kubectl apply -n opendatahub -f release/trustyai_bundle.yaml

In [ ]:
%%bash
kubectl get crd | grep trustyai
kubectl get pods -n opendatahub
kubectl get deployments -n opendatahub

---

### TrustyAI vorschalten

In [ ]:
%%bash
NAMESPACE=kserve-test
NAME=qwen-guardrails

openssl req -x509 -newkey rsa:4096 \
  -keyout /tmp/${NAME}.key \
  -out /tmp/${NAME}.crt \
  -days 365 \
  -nodes \
  -subj "/CN=127.0.0.1" \
  -addext "subjectAltName=IP:127.0.0.1,DNS:${NAME},DNS:${NAME}.${NAMESPACE},DNS:${NAME}.${NAMESPACE}.svc,DNS:${NAME}.${NAMESPACE}.svc.cluster.local,DNS:qwen-guardrails-service,DNS:qwen-guardrails-service.${NAMESPACE},DNS:qwen-guardrails-service.${NAMESPACE}.svc,DNS:qwen-guardrails-service.${NAMESPACE}.svc.cluster.local"

kubectl create secret tls ${NAME}-tls \
  --cert=/tmp/${NAME}.crt \
  --key=/tmp/${NAME}.key \
  -n ${NAMESPACE} \
  --dry-run=client -o yaml | kubectl apply -f -

GuardrailsOrchestrator starten

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: ConfigMap
metadata:
  name: qwen-guardrails-config
  namespace: kserve-test
data:
  config.yaml: |
    chat_generation:
      service:
        hostname: qwen-llm-predictor.kserve-test.svc.cluster.local
        port: 80
      # Falls KServe den registrierten Modellnamen im vLLM-Format prüft:
      model_id: "qwen" 
    detectors:
      regex:
        type: text_contents
        service:
          hostname: "127.0.0.1"
          port: 8080
        chunker_id: whole_doc_chunker
        default_threshold: 0.5
---
apiVersion: v1
kind: ConfigMap
metadata:
  name: qwen-guardrails-gateway-config
  namespace: kserve-test
data:
  config.yaml: |
    orchestrator:
      host: "qwen-guardrails-service.kserve-test.svc.cluster.local"
      port: 8032
      tls:
        cert_path: /etc/tls/private/tls.crt
    detectors:
      - name: regex
        server: regex
        input: true
        output: true
    routes:
      - name: pii
        detectors:
          - regex
---
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: GuardrailsOrchestrator
metadata:
  name: qwen-guardrails
  namespace: kserve-test
spec:
  orchestratorConfig: qwen-guardrails-config
  enableBuiltInDetectors: true
  enableGuardrailsGateway: true
  guardrailsGatewayConfig: qwen-guardrails-gateway-config
  replicas: 1
  # Zwingt die Container, das lokale Zertifikat als vertrauenswürdig einzustufen
  env:
    - name: SSL_CERT_FILE
      value: /etc/tls/private/tls.crt
EOF


In [ ]:
%%bash
kubectl patch deployment qwen-guardrails -n kserve-test --type='json' -p='[
  {
    "op": "add",
    "path": "/spec/template/spec/containers/0/securityContext",
    "value": {
      "runAsNonRoot": true,
      "runAsUser": 1001,
      "runAsGroup": 1001,
      "allowPrivilegeEscalation": false,
      "capabilities": {
        "drop": ["ALL"]
      }
    }
  }
]'

In [ ]:
%%bash
kubectl get guardrailsorchestrator -n kserve-test
kubectl get pods -n kserve-test
kubectl get svc -n kserve-test

---

### Testen

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF | tee ~/data/env-kserve-nvidia.py
OPENAI_API_KEY="kserve-nvidia"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="qwen"
AI_NAME=""
AI_IP="${AI_IP}"
AI_BASE_URL="http://localhost:30633/pii/v1"
EOF

In [ ]:
%run ~/data/env-kserve-nvidia.py
from openai import OpenAI

client = OpenAI(
    base_url=AI_BASE_URL,
    api_key="dummy",
)

response = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "user", "content": "Antworte in einem Satz: Wer war John F. Kennedy?"}
    ],
    max_tokens=80,
    temperature=0.2,
)

print(response.choices[0].message.content)